# YOLO training: **SageMaker xlarge (instance)** or **local CUDA**

Same command works in both places. **No S3** — keep `data/processed/splits/` on the instance’s disk (EBS) or your PC.

### A — AWS SageMaker (Notebook / Studio on **ml.g4dn.xlarge**)

1. Open the instance, clone or upload this repo → `cd PROJECTS/CrowdNav`
2. `pip install -r requirements.txt`
3. Put data under `data/processed/splits/` (`data.yaml` with `path: .`)
4. Run the shell in the next section. **Omit `--device`** to auto-pick GPU if CUDA is visible; or use `--device 0` explicitly.

### B — Local machine with **NVIDIA CUDA**

1. Same `cd PROJECTS/CrowdNav`, `pip install -r requirements.txt`
2. Same data layout
3. Run the same command. Use `--device 0` or leave default (auto: CUDA if available, else CPU)

**Overrides:** `CROWDNAV_DEVICE=cpu` forces CPU. `--device cpu` does the same.

**Optional:** ClearML from `../.env`. **Docker:** `deploy/Dockerfile` + `docker-compose.yml`.

In [ ]:
# Core: local training + optional ClearML. SageMaker is optional (install only if needed).
%pip install python-dotenv clearml -q

import os
from dotenv import load_dotenv

load_dotenv('../.env')
print("CLEARML keys set:", bool(os.environ.get("CLEARML_API_ACCESS_KEY")))

### 1. Data on disk

- **SageMaker (ml.g4dn.xlarge):** keep `data/processed/splits/` on the instance (upload in Jupyter, `git` + DVC, or `scp`).
- **Local:** same path. `data.yaml` with `path: .` stays portable.
- **Hardware:** this repo defaults (`batch=16`, `workers=4`) are sized for **ml.g4dn.xlarge** (NVIDIA T4, 16 GB GPU VRAM, 16 GB system RAM). Lower `--batch` if you hit OOM.

In [ ]:
### 2. Run training (SageMaker instance **or** local — same command)

From `PROJECTS/CrowdNav`:

```bash
python scripts/train_yolo.py \
  --data-yaml data/processed/splits/data.yaml \
  --epochs 100 --batch 16 --workers 4 --model-cfg yolov8m.pt \
  --name crowdnav_run1
```

- **Auto device:** omit `--device` → uses GPU if CUDA is available (SageMaker or local), else CPU.
- **Force GPU:** `--device 0`
- **Force CPU (debug):** `--device cpu` or `export CROWDNAV_DEVICE=cpu`

Jupyter: `!` prefix or `%%bash`.

### 3. Optional: other data locations

If data lives elsewhere, pass the absolute path: `--data-yaml /path/to/splits/data.yaml` (same `train/val/test` layout). Training jobs with `/opt/ml/input/...` mounts work the same way.

In [ ]:
**Summary:** default workflow = train **on a SageMaker GPU notebook (e.g. ml.g4dn.xlarge)** or **locally** with data on disk. `train_yolo.py` prints `runtime=...` and the resolved `device` at startup.